# KuPPA Smart Waste Bin AI - Fill Level Prediction

This notebook trains a Machine Learning model (Random Forest / Linear Regression) to predict how full a trash bin will be based on factors like time, temperature, and previous usage.

**Instructions:**
1. Run the cells sequentially.
2. The final cell will output the mathematical weights/coefficients of the trained AI.
3. Copy those weights to the Node.js backend of KuPPA.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 1. Generate a simulated IoT Sensor Dataset for Smart Bins
np.random.seed(42)
n_samples = 1000

data = {
    'temperature_celsius': np.random.normal(25, 5, n_samples),
    'humidity_percent': np.random.normal(60, 15, n_samples),
    'hour_of_day': np.random.randint(0, 24, n_samples),
    'is_weekend': np.random.choice([0, 1], n_samples, p=[0.71, 0.29]),
    'previous_fill_level': np.random.uniform(0, 80, n_samples)
}

df = pd.DataFrame(data)

# Target variable: Current Fill Level (%)
# Let's assume higher temperature and weekends increase waste generation.
df['current_fill_level'] = (
    df['previous_fill_level'] + 
    (df['temperature_celsius'] * 0.5) + 
    (df['is_weekend'] * 15) + 
    (np.sin(df['hour_of_day'] / 24 * 2 * np.pi) * 10) + # Peaks around midday
    np.random.normal(0, 5, n_samples) # Noise
)

# Cap at 100%
df['current_fill_level'] = df['current_fill_level'].clip(0, 100)

print("Sample Generated Data:")
print(df.head())

In [ ]:
# 2. Train the AI Model
X = df.drop('current_fill_level', axis=1)
y = df['current_fill_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"Model Root Mean Squared Error (RMSE): {rmse:.2f}%")

In [ ]:
# 3. Extract the Trained Weights for Node.js Integration
# This allows us to use the AI mathematically in Javascript without running a Python server.
weights = model.coef_
bias = model.intercept_

print("\n--- EXPORTED AI MODEL WEIGHTS FOR NODE.JS ---")
print(f"const bias = {bias:.4f};")
print(f"const weights = {{")
for feature, weight in zip(X.columns, weights):
    print(f"  '{feature}': {weight:.4f},")
print("};")
print("\n// Add this to KuPPA backend!")